# 04 — FashionCLIP transfer experiment

This notebook demonstrates the FashionCLIP backend on a **local, licensed fashion retrieval manifest**. It intentionally does not bundle DeepFashion images or labels. The public INQUIRE dataset is about natural-world imagery, so FashionCLIP on INQUIRE is an *out-of-domain transfer diagnostic*, not a valid in-domain fashion evaluation.

Prepare a CSV where each row is one query–candidate pair, with columns:

- `query`: text query, such as `red midi dress with long sleeves`
- `image_path`: local image path (absolute or relative to the manifest)
- `relevant`: binary relevance judgment (0/1)
- `initial_score`: score produced by the first-stage retrieval method
- optional `image_id`, `category`, `color`, `style` columns for analysis

Only use images and annotations under a license that permits your intended research. DeepFashion usage conditions can restrict commercial use; check the current official terms.

## 0. Initial Setup

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

In [5]:
ROOT = Path.cwd()
if not (ROOT / "src" / "image_rerank").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from image_rerank.data import iter_local_manifest
from image_rerank.embeddings import EncoderSettings, build_encoder
from image_rerank.experiment import prepare_cached_embeddings, load_cached_split, evaluate_queries
from image_rerank.plots import plot_tradeoff, plot_metric_bars
from image_rerank.artifacts import ensure_dir

MANIFEST_PATH = ROOT / "data" / "fashion_manifest.csv"
IMAGE_ROOT = None  # Set to a directory if image_path values are relative to a separate root.
MAX_QUERIES = None   # Set None for a full fashion experiment.
CACHE_ROOT = ROOT / "artifacts" / "cache"
OUTPUT_DIR = ROOT / "artifacts" / "experiments" / "fashionclip_local"
RUN_NAME = "fashionclip_local"

if not MANIFEST_PATH.exists():
    print(f"Manifest not found: {MANIFEST_PATH}")
    print("Create it using the schema described in the Markdown cell, then rerun this notebook.")
else:
    print(pd.read_csv(MANIFEST_PATH).head())

Manifest not found: d:\01 Work\14-research-Image-MMR\data\fashion_manifest.csv
Create it using the schema described in the Markdown cell, then rerun this notebook.


## 1. Initialize FashionCLIP

This uses the Hugging Face checkpoint `patrickjohncyh/fashion-clip`. The implementation freezes the model, normalizes image/text embeddings, and uses a batch size conservative for 4 GB VRAM. The first call downloads model weights; it does not fine-tune them.

In [4]:
if MANIFEST_PATH.exists():
    settings = EncoderSettings(
        backend="fashionclip",
        model_name="patrickjohncyh/fashion-clip",
        device="auto",
        batch_size=4,
        use_amp=True,
    )
    encoder = build_encoder(settings)
    print("Encoder cache key:", encoder.cache_key)
    print("Resolved device:", encoder.device)
else:
    encoder = None

## 2. Embed/cache the local candidate lists

Query cache files are keyed by the dataset split label and encoder key. Use a new split label such as `fashion_local` so these embeddings are not mixed with INQUIRE caches.

In [ ]:
if MANIFEST_PATH.exists():
    paths, cache_stats = prepare_cached_embeddings(
        iter_local_manifest(str(MANIFEST_PATH), image_root=IMAGE_ROOT, max_queries=MAX_QUERIES),
        encoder,
        split="fashion_local",
        cache_root=CACHE_ROOT,
    )
    print(cache_stats)
    print("Saved query cache files:", len(paths))

## 3. Evaluate and persist fashion results

All methods see each query's same candidate shortlist. Compare initial retrieval ranking, FashionCLIP query–image similarity, fixed MMR, adaptive MMR, greedy DPP, and cluster-xQuAD. Inspect both relevance and diversity; in product retrieval, diverse wrong products are not an improvement.

In [ ]:
if MANIFEST_PATH.exists():
    fashion_queries = load_cached_split(CACHE_ROOT, "fashion_local", encoder.cache_key)
    if MAX_QUERIES is not None:
        fashion_queries = fashion_queries[:MAX_QUERIES]
    metrics_df, summary_df = evaluate_queries(
        fashion_queries,
        top_k=50,
        diversity_k=10,
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,
    )
    figure_dir = ensure_dir(OUTPUT_DIR / "figures")
    tradeoff_path = plot_tradeoff(metrics_df, figure_dir / "fashion_relevance_diversity_latency.png")
    bar_path = plot_metric_bars(summary_df, figure_dir / "fashion_metric_comparison.png")
    display(summary_df.round(4))
    display(Image(filename=str(tradeoff_path)))
    display(Image(filename=str(bar_path)))

## Limitations and reporting

- The manifest's relevance judgments and initial scores define the task; document exactly how they were created.
- If the query texts are derived from attributes, avoid using the same ground-truth attribute labels as both input and evaluation target without explicitly stating this design.
- Use query-level splits so the same product/query family does not leak between calibration and test.
- Report per-query paired confidence intervals, p50/p95 latency, and the chosen embedding checkpoint revision.
- A FashionCLIP improvement over CLIP may come from its domain adaptation, not from the reranker. Keep `encoder_similarity` in every table to separate those effects.